In [ ]:
import pandas as pd
from pathlib import Path


######### EXTRACTION #########
print('Checking source data.......\n')
# check if csv formatted source data exist
if not Path('Source_Data/Online_Retail.csv').exists():

    print('CSV Source data does not exists. coverting original...')

    # creating datafram
    df = pd.read_excel('Source_Data/Online_Retail.xlsx')
    # covert data source to .csv
    df.to_csv('Source_Data/Online_Retail.csv', index=False)

else:
    df = pd.read_csv('Source_Data/Online_Retail.csv')
    print(df.info())

In [ ]:
###### Stage Data #####
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# load environment variable from .env
load_dotenv()


db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)

# load data to the staging table
df.to_sql('retail_data_staging', engine, if_exists='replace', index=False)

In [ ]:
########## Transformation #############

# Get staged data
staged_df = pd.read_sql_table('retail_data_staging', engine)

In [ ]:
# Separate rows with null CustomerID
null_customerid_df = staged_df[staged_df['CustomerID'].isnull()]

# Save null records to a separate table for audit/review
null_customerid_df.to_sql('retail_data_rejected', 
                          engine, 
                          if_exists='replace', 
                          index=False)

print(f"Rejected records with null CustomerID: {len(null_customerid_df)}")

In [ ]:
# Keep only rows with valid CustomerID for transformation
staged_df = staged_df[staged_df['CustomerID'].notna()]

print(f"Valid records for processing: {len(staged_df)}")

# convert CustomerID from float to Str
staged_df['CustomerID'] = staged_df['CustomerID'].astype(int)
# convert Datatime from str to datetime 
staged_df['InvoiceDate'] = pd.to_datetime(staged_df['InvoiceDate'])
staged_df.columns = staged_df.columns.str.lower()

In [ ]:
# After all transformations, create a surrogate key as an ID
staged_df.insert(0, 'transaction_id', range(1000, 1000 + len(staged_df)))

staged_df.info()

In [ ]:
# Keep returns separate for analysis
valid_df = staged_df[(staged_df['quantity'] > 0) & (staged_df['unitprice'] > 0)].copy()
returns_df = staged_df[staged_df['quantity'] < 0].copy()




In [ ]:
##### Load cleaned data to the Database
# Save transformed cleaned stage data
valid_df.to_sql('retail_data_clean', engine, if_exists='replace', index=False)
print('Cleaned data saved')

returns_df.to_sql('retail_data_returns', engine, if_exists='replace', index=False)
